# FedAvg GĐ2 — Scratch Protocol v4 (FedAvg-only)
Protocol `stage2_scratch_fedavg_v4`. Scratch MobileNetV3-Small 38 lớp, `weights=None` (pretrained=False).
GĐ2 CHỈ TRAIN FEDAVG. Tuyệt đối không huấn luyện lại Centralized hoặc Local-only trong GĐ2.
Kết quả Centralized/Local-only chỉ đọc từ GĐ1 làm tham chiếu lịch sử (historical_reference, strict_comparison_eligible=False).
Mặc định preflight. Các actions: `preflight`, `smoke`, `pilot`, `run`, `collect`, `compare-stage1`, `flower_verify`.


In [ ]:
from pathlib import Path
import json, shutil, subprocess, sys, os, hashlib

print("Contents of /kaggle/input:", [p.name for p in Path('/kaggle/input').iterdir()])

# Resolved input paths for Kaggle environment
candidates = list(Path('/kaggle/input').glob('**/stage2_scratch/experiment.py'))
assert len(candidates) == 1, f"Expected exactly 1 package with stage2_scratch/experiment.py, found {len(candidates)}: {candidates}"
PACKAGE_SOURCE = candidates[0].parent.parent
print(f"PACKAGE_SOURCE: {PACKAGE_SOURCE}")

candidate_datasets = [
    Path('/kaggle/input/plantvillage-raw-images/raw/color'),
    Path('/kaggle/input/plantvillage-raw-images/color'),
    *Path('/kaggle/input').glob('**/raw/color'),
    *Path('/kaggle/input').glob('**/color'),
]
valid_datasets = [d for d in candidate_datasets if d.is_dir() and len([p for p in d.iterdir() if p.is_dir()]) == 38]
seen = set()
unique_datasets = [d for d in valid_datasets if not (d in seen or seen.add(d))]
assert len(unique_datasets) == 1, f"Expected exactly 1 valid DATASET_ROOT with 38 classes, found {len(unique_datasets)}: {unique_datasets}"
DATASET_ROOT = unique_datasets[0]
print(f"DATASET_ROOT: {DATASET_ROOT}")

WORK = Path('/kaggle/working/scratch_package')
OUTPUT = Path('/kaggle/working/stage2_scratch_v4')
RESUME = False  # Only True to continue this same scratch experiment
RESUME_SOURCE = None  # Path('/kaggle/input/previous-output/stage2_scratch_v4')
ACTION = 'run'
CONDITIONS = ['label100', 'label1', 'label01', 'quantity100', 'quantity01', 'label_quantity01', 'feature100', 'feature01']
SEEDS = [42]
OPTIMIZER = 'sgd'  # sgd (Algorithm 1)

# Set action-specific session limits
if ACTION == 'smoke':
    SESSION_MINUTES = 5
elif ACTION == 'pilot':
    SESSION_MINUTES = 45
else:
    SESSION_MINUTES = 420

assert (PACKAGE_SOURCE / 'stage2_scratch/experiment.py').is_file()
assert DATASET_ROOT.is_dir()
assert ACTION in {'preflight', 'smoke', 'pilot', 'run', 'collect', 'compare-stage1', 'flower_verify'}
print(f"Configured ACTION={ACTION}, SESSION_MINUTES={SESSION_MINUTES}")


In [ ]:
# Copy only code and frozen manifests according to allowlist
folders = ['stage2_scratch', 'stage2_matched', 'stage1_compat', 'src', 'fl_training', 'tests', 'data/partitions_stage2_scratch_v3']
for folder in folders:
    source, target = PACKAGE_SOURCE / folder, WORK / folder
    assert source.is_dir(), f"Missing folder: {source}"
    files = [p for p in source.rglob('*') if p.is_file() and '__pycache__' not in p.parts and p.suffix != '.pyc']
    expected = {p.relative_to(source).as_posix() for p in files}
    if target.exists():
        actual = {p.relative_to(target).as_posix() for p in target.rglob('*')
                  if p.is_file() and '__pycache__' not in p.parts and p.suffix != '.pyc'}
        assert actual == expected, 'Stale WORK files: use a new WORK path'
    for p in files:
        dest = target / p.relative_to(source)
        if dest.exists():
            assert dest.read_bytes() == p.read_bytes(), f'Stale code/data: {dest}'
        else:
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(p, dest)

# Copy root files
root_files = ['data/duplicate_review.json', 'data/visually_verified_pairs.json', 'data/four_visually_verified_pairs.json',
              'kaggle_stage2_scratch_v4.ipynb', 'requirements-stage1.txt', 'requirements-stage2-matched.txt', 'pyproject.toml']
for rf in root_files:
    src_f, dst_f = PACKAGE_SOURCE / rf, WORK / rf
    if src_f.is_file():
        dst_f.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src_f, dst_f)

# Verify release manifest v4
sys.path.insert(0, str(WORK))
from fl_training.package_verify import verify_package_manifest

manifest_path = PACKAGE_SOURCE / 'release_manifest_v4.json'
assert manifest_path.is_file(), f"Release manifest missing: {manifest_path}"
verify_res = verify_package_manifest(WORK, manifest_path=manifest_path)
assert verify_res['verified_files'] > 0, 'No files verified in package'
print(f"Package integrity verified: {verify_res['verified_files']} files checked successfully.")
print(f"Manifest SHA256: {verify_res['manifest_sha256']}")

if ACTION == 'run':
    if not RESUME:
        if RESUME_SOURCE is not None or OUTPUT.exists():
            raise RuntimeError('Fresh run requires a new OUTPUT and RESUME_SOURCE=None')
    elif RESUME_SOURCE is not None:
        if OUTPUT.exists():
            raise RuntimeError('Restore target must not exist; choose a new OUTPUT')
        if not (RESUME_SOURCE / 'scratch_protocol.json').is_file():
            raise RuntimeError('Only scratch outputs can be restored')
        shutil.copytree(RESUME_SOURCE, OUTPUT)
    elif not (OUTPUT / 'scratch_protocol.json').is_file():
        raise RuntimeError('Resume requires an existing scratch output or RESUME_SOURCE')
os.chdir(WORK)


In [ ]:
# Verify or install compatible PyTorch and torchvision
def check_versions():
    res = subprocess.run(
        [sys.executable, '-c', 'import torch, torchvision; print(torch.__version__.split("+")[0], torchvision.__version__.split("+")[0])'],
        capture_output=True, text=True
    )
    if res.returncode != 0:
        return False, False, False, f"error: {res.stderr.strip()}"
    parts = res.stdout.strip().split()
    if len(parts) != 2:
        return False, False, False, f"unexpected: {res.stdout.strip()}"
    t_ver, v_ver = parts
    return t_ver.startswith('2.6.'), v_ver.startswith('0.21.'), True, f"torch={t_ver}, torchvision={v_ver}"

t_ok, v_ok, f_ok, current_vers = check_versions()
print(f"Initial environment: {current_vers}")

if not (t_ok and v_ok and f_ok):
    print("Installing dependencies (PyTorch 2.6.x, torchvision 0.21.x)...")
    has_gpu = False
    try:
        res = subprocess.run(['nvidia-smi'], capture_output=True)
        has_gpu = (res.returncode == 0)
    except Exception:
        has_gpu = False
    idx = 'https://download.pytorch.org/whl/cu124' if has_gpu else 'https://download.pytorch.org/whl/cpu'
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--disable-pip-version-check',
                    'torch==2.6.0', 'torchvision==0.21.0', '--index-url', idx], check=True)
    t_ok, v_ok, f_ok, current_vers = check_versions()
    print(f"Post-install environment: {current_vers}")

assert t_ok and v_ok and f_ok, f"Incompatible versions after bootstrap: {current_vers}"

res = subprocess.run(
    [sys.executable, '-c', 'import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)'],
    capture_output=True, text=True, check=True
)
print("Runtime info:", res.stdout.strip())
if ACTION == 'run':
    assert 'True' in res.stdout.split(), 'Enable GPU accelerator before a full run'

suite = json.loads((WORK / 'data/partitions_stage2_scratch_v3/suite.json').read_text())
assert suite['smoke'] is False and suite['num_clients'] == 5
print({'counts': suite['counts'], 'conditions': CONDITIONS, 'seeds': SEEDS,
       'rounds': 10, 'local_epochs': 1, 'session_minutes': SESSION_MINUTES,
       'pretrained': False, 'initialization': 'random', 'resume': RESUME})

# Verify imported module path is inside WORK
import stage2_scratch.experiment as exp
assert Path(exp.__file__).resolve().is_relative_to(WORK.resolve()), f"Module not imported from WORK: {exp.__file__}"
print(f"Module verification: loaded from WORK correctly with protocol {exp.PROTOCOL}.")


In [ ]:
# Verify the same production runner on a hash-checked subset before full-data preflight/training.
if ACTION == 'run':
    GPU_SMOKE = Path('/kaggle/working/fedavg_gpu_smoke_v4')
    SMOKE_SCRIPT = "from pathlib import Path\nimport hashlib,json,os,sys,time\nimport torch\nfrom stage2_matched.data import ManifestDataset,read_rows\nfrom stage2_scratch.experiment import make_job,sample_whole_groups_per_class,record_initialization,SentinelTestDataset\nfrom stage2_scratch.runner import run_single_fedavg_job\nfrom stage1_compat.budget import BudgetLedger\ntorch.set_num_threads(4)\ndevice=torch.device(sys.argv[3])\nif device.type=='cuda': assert torch.cuda.is_available(), 'CUDA unavailable'\ndata=Path(sys.argv[1]); output=Path(sys.argv[2]); output.mkdir(parents=True,exist_ok=False)\nsuite_path=Path('data/partitions_stage2_scratch_v3')\nsuite=json.loads((suite_path/'suite.json').read_text())\ncondition=suite_path/'label100'\ntrain_rows=sample_whole_groups_per_class(read_rows(condition/'centralized_train.csv'),2,42)\nval_rows=sample_whole_groups_per_class(read_rows(condition/'global_val.csv'),1,42)\ninventory=json.loads((condition/'image_content.json').read_text())\nfor row in train_rows+val_rows:\n    rel=row['relative_path']\n    assert hashlib.sha256((data/rel).read_bytes()).hexdigest()==inventory[rel], rel\ntrain=ManifestDataset(train_rows,data,True)\nval=ManifestDataset(val_rows,data,False)\npartitions=[[i for i,r in enumerate(train_rows) if r['client_id']==cid] for cid in range(5)]\nassert all(partitions)\njob=make_job('label100',42,smoke=True,rounds=2,optimizer='sgd')\ntrain.initialization_sha256=record_initialization(job,output)\ntrain.identity_context={'scope':'stage2_scratch_fedavg_v4','diagnostic':True,'backend':'sequential','initialization':'random','pretrained':False}\nledger=BudgetLedger(output,user_quota_hours=1)\nos.environ['STAGE1_SOFT_DEADLINE']=str(time.time()+300)\ntry:\n    result=run_single_fedavg_job(job,train,val,SentinelTestDataset(),suite['class_names'],partitions,output,ledger,device,resume=False,calibration=True)\nfinally:\n    os.environ.pop('STAGE1_SOFT_DEADLINE',None)\nassert result['status']=='CALIBRATED_NO_TEST' and result['test_evaluated'] is False\nprint('GPU_SMOKE_RUNNER_PASSED',json.dumps({'device':str(device),'train':len(train_rows),'val':len(val_rows),'rounds':2}))\n"
    subprocess.run([sys.executable, '-u', '-c', SMOKE_SCRIPT,
        str(DATASET_ROOT), str(GPU_SMOKE), 'cuda'], check=True, timeout=360)
    smoke_metrics = list(GPU_SMOKE.rglob('calibration_metrics.json'))
    assert smoke_metrics, 'GPU smoke produced no calibration evidence'
    for path in smoke_metrics:
        evidence = json.loads(path.read_text())
        assert evidence['status'] == 'CALIBRATED_NO_TEST' and evidence['test_evaluated'] is False
    Path('/kaggle/working/GPU_SMOKE_PASSED.json').write_text(json.dumps({
        'status': 'GPU_SMOKE_PASSED', 'manifest_sha256': hashlib.sha256((PACKAGE_SOURCE / 'release_manifest_v4.json').read_bytes()).hexdigest(),
        'metrics': [str(p) for p in smoke_metrics]}, indent=2))
    print('GPU_SMOKE_PASSED: beginning full-data preflight and FedAvg; scratch initialization is independent of smoke.')
else:
    print(f'Skipping GPU smoke gate for ACTION={ACTION}')


In [ ]:
command = [sys.executable, '-u', '-m', 'stage2_scratch', ACTION,
           '--suite', 'data/partitions_stage2_scratch_v3',
           '--dataset', str(DATASET_ROOT), '--output', str(OUTPUT),
           '--optimizer', OPTIMIZER]
if ACTION == 'run':
    command += ['--conditions', *CONDITIONS, '--seeds', *map(str, SEEDS),
                '--session-minutes', str(SESSION_MINUTES), '--device', 'cuda']
    if RESUME:
        command += ['--resume']
elif ACTION in ('smoke', 'pilot'):
    command += ['--session-minutes', str(SESSION_MINUTES), '--device', 'cuda']
subprocess.run(command, check=True)


## Tiếp tục và tổng hợp

GĐ2 chỉ huấn luyện phân tán bằng FedAvg.
Để đối chiếu với baseline GĐ1, chạy ACTION='compare-stage1' để đọc bảng tham chiếu lịch sử (historical_reference).
Để kiểm tra các trục non-IID còn lại, đặt CONDITIONS thành:
`['quantity100', 'quantity01', 'label_quantity01', 'feature100', 'feature01']`.
Sau khi hoàn thành các runs, đặt ACTION='collect' để xuất `scratch_comparison.json`.


In [ ]:
# Summarize completed jobs; paused jobs retain their committed checkpoints.
if ACTION in ('run', 'collect'):
    subprocess.run([sys.executable, '-u', '-m', 'stage2_scratch', 'collect',
        '--suite', 'data/partitions_stage2_scratch_v3', '--output', str(OUTPUT)], check=True)
    subprocess.run([sys.executable, '-u', '-m', 'stage2_scratch', 'compare-stage1',
        '--suite', 'data/partitions_stage2_scratch_v3', '--output', str(OUTPUT)], check=True)
else:
    print(f'Skipping collect and compare for ACTION={ACTION}')
